In [17]:
using Pkg
Pkg.activate(".")
using Distributed
using CSV, DataFrames, BSON, Random

  Activating project at `c:\Users\mikul\Desktop\Persistence of Shocks\tvPersistence.jl - Cleaned`


In [2]:
# Pkg.add("Statistics")

In [18]:
rmprocs(workers())
num_workers = 4            # ← set to number of CPU cores you want to use
num_replicates = 30        # ← set as desired
addprocs(num_workers)

┌ Warning: rmprocs: process 1 not removed
└ @ Distributed C:\Users\mikul\AppData\Local\Programs\Julia-1.11.4\share\julia\stdlib\v1.11\Distributed\src\cluster.jl:1049


4-element Vector{Int64}:
 6
 7
 8
 9

In [20]:
# (these were read from example_config.txt in the original script)
@everywhere begin
    data_file                    = "PCEpi.csv"
    data_column                  = " " #1 #"x1"
    missingstring                = "NA"

    ar_order                     = 1
    in_sample_window_size        = 610
    forecast_horizon             = 1
    forecast_length              = 136
    random_seed                  = 1#234

    smoothing_bandwidth          = 0.1 #0.3# 0.1 #0.05
    cutoff_start_index           = 1

    benchmark_method             = "RW"
    comparison_method            = "tvEWD"

    tvp_kernel_width             = 0.3
    kernel_type                  = "triweight"#"Epanechnikov" #"Gaussian"#"triweight"#"Epanechnikov"
    max_ar_order                 = 2
    jmax_scale                   = 5
    ar_lag_for_trend             = 1
    tvp_constant_kernel_width    = 0.05
    irf_kernel_width             = 0.2
    forecast_kernel_width        = 0.6
    smoothing_kernel             = "triweight"#"one-sided"#"triweight"#"Epanechnikov" #
    kernel_type_tvEWD            = "triweight"#"Epanechnikov" #"Gaussian"#"triweight"#"Epanechnikov"
    kernel_type_tvHAR            = "triweight"#"Epanechnikov" #"Gaussian"#"triweight"#"Epanechnikov"
    kernel_type_tvAR             = "triweight"#"Epanechnikov" #"Gaussian"#"triweight"#"Epanechnikov"

    alpha_level                  = 0.05
end

In [21]:
const SED_PATH = abspath("src/SED_Thresholds/SEDThresholds.jl")

"c:\\Users\\mikul\\Desktop\\Persistence of Shocks\\tvPersistence.jl - Cleaned\\src\\SED_Thresholds\\SEDThresholds.jl"

In [22]:
@everywhere include($SED_PATH)        # <— absolute path shipped to workers
@everywhere using .SEDThresholds

In [23]:
# load
# df = CSV.File(data_file, missingstring=[missingstring], header=true) |> DataFrame;

df = CSV.File(data_file, missingstring=[missingstring], header=false) |> DataFrame;

# turn column name into a Symbol, drop missings & scale
# col_sym = Symbol(data_column);
# series  = Float64.(df[.!ismissing.(df[!, col_sym]), col_sym]);
series  = df[:,1].*100;

In [24]:
sed_vals = pmap(1:num_replicates) do i
    # re-seed for reproducibility
    Random.seed!(random_seed + i)

    calculate_bootstrap_threshold_parallel(
        i, series,
        ar_order, in_sample_window_size, forecast_horizon,
        smoothing_bandwidth,
        Symbol(benchmark_method), Symbol(comparison_method);
        fcast_len                  = forecast_length,
        tvp_kernel_width           = tvp_kernel_width,
        kernel_type_tvEWD          = kernel_type_tvEWD,
        kernel_type_tvHAR          = kernel_type_tvHAR,
        kernel_type_tvAR           = kernel_type_tvAR,
        smoothing_kernel           = smoothing_kernel,
        max_ar_order               = max_ar_order,
        jmax_scale                 = jmax_scale,
        ar_lag_for_trend           = ar_lag_for_trend,
        tvp_constant_kernel_width  = tvp_constant_kernel_width,
        irf_kernel_width           = irf_kernel_width,
        forecast_kernel_width      = forecast_kernel_width
    )
end;

# remove working processes
rmprocs(workers());

      From worker 7:	[ Info: Performing boostrap simulation number 2
      From worker 8:	[ Info: Performing boostrap simulation number 3
      From worker 9:	[ Info: Performing boostrap simulation number 4
      From worker 6:	[ Info: Performing boostrap simulation number 1
      From worker 7:	[ Info: Bootstrap 2 generated.
      From worker 9:	[ Info: Bootstrap 4 generated.
      From worker 8:	[ Info: Bootstrap 3 generated.
      From worker 6:	[ Info: Bootstrap 1 generated.
      From worker 7:	[ Info: Performing boostrap simulation number 5
      From worker 9:	[ Info: Performing boostrap simulation number 6
      From worker 8:	[ Info: Performing boostrap simulation number 7
      From worker 6:	[ Info: Performing boostrap simulation number 8
      From worker 9:	[ Info: Bootstrap 6 generated.
      From worker 9:	[ Info: Performing boostrap simulation number 9
      From worker 7:	[ Info: Bootstrap 5 generated.
      From worker 7:	[ Info: Performing boostrap simulation number 

In [26]:
output_bson = Dict("SED_values" => sed_vals)
BSON.bson("SED_values_new.bson", output_bson)

In [27]:
# 1) Count NaNs in each inner vector
nan_counts_per_series = map(v -> count(isnan, v), sed_vals)

# 2) Total number of NaNs across all series
total_nans = sum(nan_counts_per_series)

0

In [28]:
thr = SEDThresholds.compute_global_threshold(sed_vals, cutoff_start_index, alpha_level)
println("SED threshold: ", thr)

SED threshold: 0.012010934262352922


In [12]:
# Save the SED values into BSON file
# BSON.@save "sed_thresholds.bson" sed_vals thr